# DACOE Netflix TV Shows/Movie Watch Time Data

This notebook executes the **entire workflow in one place**:

0. Setup & Imports
1. Load Netflix data
2. Extract IMDb data via OMDb API
3. Merge IMDb into Netflix Data
4. Feature Engineering


## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import time
import os
from datetime import datetime

## 1. Load Netflix Base Dataset

In [177]:
netflix_df = pd.read_excel('data/DACOE Challenge 2026 Dataset.xlsx',engine="openpyxl")

netflix_df["Premiere"] = (
    netflix_df["Premiere"]
    .astype(str)              # kill datetime + int differences
    .str.extract(r"(\d{4})")  # extract year
    .astype("string")         # consistent dtype
)

netflix_df["Title"] = (
    netflix_df["Title"]
    .astype(str)              # kill datetime + int differences
)

netflix_df.head()

,Rank,Title,Type,Premiere,Genre,Watchtime,Watchtime in Million
0,1,The Night Agent,TV Show,2023,Action,812100000,812.1M
1,2,Ginny & Georgia,TV Show,2021,Drama,665100000,665.1M
2,3,The Glory,TV Show,2022,Thriller,622800000,622.8M
3,4,Wednesday,TV Show,2022,Fantasy,507700000,507.7M
4,5,Queen Charlotte: A Bridgerton Story,TV Show,2023,Drama,503000000,503.0M


In [178]:
netflix_df["Premiere"].apply(type).value_counts()

Premiere
<class 'str'>                         18030
<class 'pandas.api.typing.NAType'>      134
Name: count, dtype: int64

There 134 empty Premiere rows

In [179]:
netflix_df["Title"].apply(type).value_counts()

Title
<class 'str'>    18164
Name: count, dtype: int64

In [180]:
# Convert column names to lowercase and replace space with underscore
netflix_df.columns = netflix_df.columns.str.strip().str.lower().str.replace(' ', '_')
netflix_df.head()

,rank,title,type,premiere,genre,watchtime,watchtime_in_million
0,1,The Night Agent,TV Show,2023,Action,812100000,812.1M
1,2,Ginny & Georgia,TV Show,2021,Drama,665100000,665.1M
2,3,The Glory,TV Show,2022,Thriller,622800000,622.8M
3,4,Wednesday,TV Show,2022,Fantasy,507700000,507.7M
4,5,Queen Charlotte: A Bridgerton Story,TV Show,2023,Drama,503000000,503.0M


In [181]:
netflix_df.shape

(18164, 7)

In [182]:
#netflix_df.to_csv('netflix_df.csv', index = False)

## 2. Extract IMDb data via OMDb API

Ensure you get your OMDB_API_KEY from https://www.omdbapi.com.

In [100]:
#OMDB_API_KEY = os.getenv('OMDB_API_KEY')

OMDB_API_KEY = '' #Enter your OMDB_API_KEY here 
OMDB_URL = 'http://www.omdbapi.com/'

assert OMDB_API_KEY, 'OMDB_API_KEY not set'

In [101]:
# Helper Functions
def split_by_year_presence(df, year_col="year"):
    """
    Split a DataFrame into two DataFrames based on whether the year column is present.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame
    year_col : str, default="year"
        Name of the year column

    Returns
    -------
    df_with_year : pandas.DataFrame
        Rows where year is NOT empty / NaN
    df_without_year : pandas.DataFrame
        Rows where year is empty / NaN
    """

    df_with_year = df[df[year_col].notna()].copy()
    df_without_year = df[df[year_col].isna()].copy()

    return df_with_year, df_without_year




def clean_title(title):
    return (title.str.lower()
              .str.replace('&', 'and')
              .str.replace(r'[^a-z0-9 ]', '', regex=True)
              .str.strip())

def fetch_omdb(title, year):
    params = {
        'apikey': OMDB_API_KEY,
        't': title,
        'y': year,
        'r': 'json'
    }
    response = requests.get(OMDB_URL, params=params, timeout=10)
    data = response.json()
    return data if data.get('Response') == 'True' else None


In [102]:
# Get netflix data with premiere and data without premiere
# title and premiere are both require to pull data from the API

df_with_year, df_without_year = split_by_year_presence(netflix_df, year_col="premiere")

In [103]:
df_with_year.shape

(18030, 7)

In [104]:
df_with_year.info()

<class 'pandas.DataFrame'>
Index: 18030 entries, 0 to 18163
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   rank                  18030 non-null  int64 
 1   title                 18030 non-null  object
 2   type                  18030 non-null  str   
 3   premiere              18030 non-null  string
 4   genre                 17879 non-null  str   
 5   watchtime             18030 non-null  int64 
 6   watchtime_in_million  18030 non-null  str   
dtypes: int64(2), object(1), str(3), string(1)
memory usage: 1.1+ MB


In [107]:
df_with_year.head()

,rank,title,type,premiere,genre,watchtime,watchtime_in_million
0,1,The Night Agent,TV Show,2023,Action,812100000,812.1M
1,2,Ginny & Georgia,TV Show,2021,Drama,665100000,665.1M
2,3,The Glory,TV Show,2022,Thriller,622800000,622.8M
3,4,Wednesday,TV Show,2022,Fantasy,507700000,507.7M
4,5,Queen Charlotte: A Bridgerton Story,TV Show,2023,Drama,503000000,503.0M


In [91]:
df_without_year.shape

(134, 7)

In [108]:
df_without_year.head()

,rank,title,type,premiere,genre,watchtime,watchtime_in_million
591,592,PokÃ©mon: Ultimate Journeys,TV Show,<NA>,Animation,31300000,31.3M
983,984,PokÃ©mon: Ultimate Journeys,TV Show,<NA>,Animation,21700000,21.7M
2931,2982,PokÃ©mon: Ultimate Journeys,TV Show,<NA>,Animation,6300000,6.3M
3544,3595,Badanamu Pop,TV Show,<NA>,Animation,5000000,5.0M
3930,3981,Story Time Book: Read-Along,TV Show,<NA>,Animation,4200000,4.2M


Shows/Movies with same attributes appear multiple times in the dataset but with different watch time, this is most likely because shows with multiple seasons were captured differently, unfortunately no season column was provided with the data, so a sum of the watchtime was done and a new ranking score created

In [109]:
df_with_year["premiere"].apply(type).value_counts()

premiere
<class 'str'>    18030
Name: count, dtype: int64

In [116]:
#Aggregate watchtime across seasons based on (Title + Premiere uniquely identify a show/movie)

aggregated_df = (df_with_year.groupby(["title", "premiere", "type", "genre"],
                                      as_index=False, sort=False).agg(total_watchtime=("watchtime", "sum")))


In [117]:
aggregated_df.head()

,title,premiere,type,genre,total_watchtime
0,The Night Agent,2023,TV Show,Action,812100000
1,Ginny & Georgia,2021,TV Show,Drama,967200000
2,The Glory,2022,TV Show,Thriller,622800000
3,Wednesday,2022,TV Show,Fantasy,507700000
4,Queen Charlotte: A Bridgerton Story,2023,TV Show,Drama,503000000


In [118]:
#Create a watchtime-in-millions column
aggregated_df["total_watchtime_million"] = (
    aggregated_df["total_watchtime"] / 1000000
)


In [119]:
aggregated_df.head()

,title,premiere,type,genre,total_watchtime,total_watchtime_million
0,The Night Agent,2023,TV Show,Action,812100000,812.1
1,Ginny & Georgia,2021,TV Show,Drama,967200000,967.2
2,The Glory,2022,TV Show,Thriller,622800000,622.8
3,Wednesday,2022,TV Show,Fantasy,507700000,507.7
4,Queen Charlotte: A Bridgerton Story,2023,TV Show,Drama,503000000,503.0


In [120]:
#Create a new ranking based on total watchtime
aggregated_df = (
    aggregated_df
    .sort_values("total_watchtime", ascending=False)
    .reset_index(drop=True)
)

aggregated_df["new_rank"] = aggregated_df.index + 1

In [121]:
aggregated_df.head()

,title,premiere,type,genre,total_watchtime,total_watchtime_million,new_rank
0,Ginny & Georgia,2021,TV Show,Drama,967200000,967.2,1
1,The Night Agent,2023,TV Show,Action,812100000,812.1,2
2,You,2018,TV Show,Crime,766300000,766.3,3
3,Outer Banks,2020,TV Show,Drama,740400000,740.4,4
4,The Walking Dead,2010,TV Show,Horror,738600000,738.6,5


In [122]:
# Reorder columns for clarity

aggregated_df = aggregated_df[
    [
        "new_rank",
        "title",
        "type",
        "premiere",
        "genre",
        "total_watchtime",
        "total_watchtime_million"
    ]
]


In [123]:
aggregated_df.head()

,new_rank,title,type,premiere,genre,total_watchtime,total_watchtime_million
0,1,Ginny & Georgia,TV Show,2021,Drama,967200000,967.2
1,2,The Night Agent,TV Show,2023,Action,812100000,812.1
2,3,You,TV Show,2018,Crime,766300000,766.3
3,4,Outer Banks,TV Show,2020,Drama,740400000,740.4
4,5,The Walking Dead,TV Show,2010,Horror,738600000,738.6


In [124]:
aggregated_df.shape

(14616, 7)

In [126]:
#Get data from OMDB API
imdb_rows = []
for _, r in aggregated_df[['title','premiere']].drop_duplicates().iterrows():
    record = fetch_omdb(r['title'], int(r['premiere']))
    if record:
        imdb_rows.append({
            'imdb_title': r['title'],
            'year': r['premiere'],
            'released_date': record.get('Released'),
            'rated': record.get('Rated'),
            'imdb_genre': record.get('Genre'),
            'director': record.get('Director'),
            'writer': record.get('Writer'),
            'actors': record.get('Actors'),
            'plot': record.get('Plot'),
            'imdb_type': record.get('Type'),
            'total_seasons': record.get('totalSeasons'),
            'imdb_rating': float(record['imdbRating']) if record['imdbRating']!='N/A' else None,
            'imdb_votes': int(record['imdbVotes'].replace(',','')) if record['imdbVotes']!='N/A' else None,
            'runtime_minutes': (record['Runtime'].replace(' min','')) if record['Runtime']!='N/A' else None, #int
            'awards': record.get('Awards'),
            'language': record.get('Language'),
            'country': record.get('Country'),
            'poster': record.get('Poster'),
        })
    time.sleep(0.25)

In [127]:
imdb_df = pd.DataFrame(imdb_rows)
imdb_df['title_clean'] = clean_title(imdb_df['imdb_title'])
imdb_df.head()

,imdb_title,year,released_date,rated,imdb_genre,director,writer,actors,plot,imdb_type,total_seasons,imdb_rating,imdb_votes,runtime_minutes,awards,language,country,poster,title_clean
0,Ginny & Georgia,2021,24 Feb 2021,TV-14,"Comedy, Drama",N/A,Sarah Lampert,"Brianne Howey, Antonia Gentry, Diesel La Torraca",Angsty 15-year-old Ginny Miller often feels mo...,series,3,7.5,103432.0,NaN,Nominated for 1 Primetime Emmy. 5 wins & 8 nom...,English,"United States, Canada",https://m.media-amazon.com/images/M/MV5BNGY4Nj...,ginny and georgia
1,The Night Agent,2023,23 Mar 2023,TV-MA,"Action, Drama, Thriller",N/A,N/A,"Gabriel Basso, Fola Evans-Akingbola, Luciane B...",Low-level FBI agent Peter Sutherland works in ...,series,3,7.4,151290.0,NaN,2 wins & 8 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMDU5Zj...,the night agent
2,You,2018,09 Sep 2018,TV-MA,"Crime, Drama, Romance",N/A,"Greg Berlanti, Sera Gamble","Penn Badgley, Victoria Pedretti, Charlotte Rit...",A charming and intense young man inserts himse...,series,5,7.6,361631.0,45,4 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMTVlYm...,you
3,Outer Banks,2020,15 Apr 2020,TV-MA,"Action, Crime, Drama",N/A,"Shannon Burke, Jonas Pate, Josh Pate","Chase Stokes, Madelyn Cline, Madison Bailey","On an island of haves and have-nots, teen John...",series,5,7.5,104906.0,50,5 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BY2FmYm...,outer banks
4,The Walking Dead,2010,31 Oct 2010,TV-MA,"Drama, Horror, Thriller",N/A,Frank Darabont,"Andrew Lincoln, Norman Reedus, Melissa McBride",Sheriff Deputy Rick Grimes wakes up from a com...,series,11,8.1,1180953.0,44,Won 2 Primetime Emmys. 86 wins & 237 nominatio...,English,United States,https://m.media-amazon.com/images/M/MV5BYWQwMG...,the walking dead


In [128]:
imdb_df.shape

(11928, 19)

## 3. Merge IMDb into Netflix Data

In [129]:
aggregated_df['title_clean'] = clean_title(aggregated_df['title'])

enriched_df = aggregated_df.merge(imdb_df, left_on=['title_clean','premiere'], right_on=['title_clean','year'], how='left')
#df = df.merge(trends[['title_clean','trends_peak','trends_mean']], on='title_clean', how='left')

enriched_df.head()

,new_rank,title,type,premiere,genre,total_watchtime,total_watchtime_million,title_clean,imdb_title,year,...,plot,imdb_type,total_seasons,imdb_rating,imdb_votes,runtime_minutes,awards,language,country,poster
0,1,Ginny & Georgia,TV Show,2021,Drama,967200000,967.2,ginny and georgia,Ginny & Georgia,2021,...,Angsty 15-year-old Ginny Miller often feels mo...,series,3,7.5,103432.0,NaN,Nominated for 1 Primetime Emmy. 5 wins & 8 nom...,English,"United States, Canada",https://m.media-amazon.com/images/M/MV5BNGY4Nj...
1,2,The Night Agent,TV Show,2023,Action,812100000,812.1,the night agent,The Night Agent,2023,...,Low-level FBI agent Peter Sutherland works in ...,series,3,7.4,151290.0,NaN,2 wins & 8 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMDU5Zj...
2,3,You,TV Show,2018,Crime,766300000,766.3,you,You,2018,...,A charming and intense young man inserts himse...,series,5,7.6,361631.0,45,4 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMTVlYm...
3,4,Outer Banks,TV Show,2020,Drama,740400000,740.4,outer banks,Outer Banks,2020,...,"On an island of haves and have-nots, teen John...",series,5,7.5,104906.0,50,5 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BY2FmYm...
4,5,The Walking Dead,TV Show,2010,Horror,738600000,738.6,the walking dead,The Walking Dead,2010,...,Sheriff Deputy Rick Grimes wakes up from a com...,series,11,8.1,1180953.0,44,Won 2 Primetime Emmys. 86 wins & 237 nominatio...,English,United States,https://m.media-amazon.com/images/M/MV5BYWQwMG...


In [130]:
enriched_df.to_csv('data/output.csv', index = False)

## 4. Feature Engineering

In [135]:
CURRENT_YEAR = datetime.now().year
enriched_df['premiere'] = enriched_df['premiere'].astype("int64")


enriched_df['years_since_release'] = CURRENT_YEAR - enriched_df['premiere']
enriched_df['log_watchtime'] = np.log1p(enriched_df['total_watchtime'])
enriched_df['recency_weighted_watchtime'] = enriched_df['total_watchtime'] / (enriched_df['years_since_release']+1)
enriched_df['evergreen_score'] = enriched_df['recency_weighted_watchtime'] * (enriched_df['years_since_release']+1)/1000000

In [136]:
enriched_df.head()

,new_rank,title,type,premiere,genre,total_watchtime,total_watchtime_million,title_clean,imdb_title,year,...,imdb_votes,runtime_minutes,awards,language,country,poster,years_since_release,log_watchtime,recency_weighted_watchtime,evergreen_score
0,1,Ginny & Georgia,TV Show,2021,Drama,967200000,967.2,ginny and georgia,Ginny & Georgia,2021,...,103432.0,NaN,Nominated for 1 Primetime Emmy. 5 wins & 8 nom...,English,"United States, Canada",https://m.media-amazon.com/images/M/MV5BNGY4Nj...,5,20.689916,1.612000e+08,967.2
1,2,The Night Agent,TV Show,2023,Action,812100000,812.1,the night agent,The Night Agent,2023,...,151290.0,NaN,2 wins & 8 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMDU5Zj...,3,20.515134,2.030250e+08,812.1
2,3,You,TV Show,2018,Crime,766300000,766.3,you,You,2018,...,361631.0,45,4 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMTVlYm...,8,20.457084,8.514444e+07,766.3
3,4,Outer Banks,TV Show,2020,Drama,740400000,740.4,outer banks,Outer Banks,2020,...,104906.0,50,5 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BY2FmYm...,6,20.422701,1.057714e+08,740.4
4,5,The Walking Dead,TV Show,2010,Horror,738600000,738.6,the walking dead,The Walking Dead,2010,...,1180953.0,44,Won 2 Primetime Emmys. 86 wins & 237 nominatio...,English,United States,https://m.media-amazon.com/images/M/MV5BYWQwMG...,16,20.420267,4.344706e+07,738.6


In [137]:
enriched_df['watchtime_zscore'] = (enriched_df['total_watchtime']-enriched_df['total_watchtime'].min())/(enriched_df['total_watchtime'].max()-enriched_df['total_watchtime'].min())

enriched_df['imdb_rating_zscore'] = (enriched_df['imdb_rating']-enriched_df['imdb_rating'].min())/(enriched_df['imdb_rating'].max()-enriched_df['imdb_rating'].min())

In [138]:
enriched_df.head()

,new_rank,title,type,premiere,genre,total_watchtime,total_watchtime_million,title_clean,imdb_title,year,...,awards,language,country,poster,years_since_release,log_watchtime,recency_weighted_watchtime,evergreen_score,watchtime_zscore,imdb_rating_zscore
0,1,Ginny & Georgia,TV Show,2021,Drama,967200000,967.2,ginny and georgia,Ginny & Georgia,2021,...,Nominated for 1 Primetime Emmy. 5 wins & 8 nom...,English,"United States, Canada",https://m.media-amazon.com/images/M/MV5BNGY4Nj...,5,20.689916,1.612000e+08,967.2,1.000000,0.715909
1,2,The Night Agent,TV Show,2023,Action,812100000,812.1,the night agent,The Night Agent,2023,...,2 wins & 8 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMDU5Zj...,3,20.515134,2.030250e+08,812.1,0.839624,0.704545
2,3,You,TV Show,2018,Crime,766300000,766.3,you,You,2018,...,4 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMTVlYm...,8,20.457084,8.514444e+07,766.3,0.792266,0.727273
3,4,Outer Banks,TV Show,2020,Drama,740400000,740.4,outer banks,Outer Banks,2020,...,5 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BY2FmYm...,6,20.422701,1.057714e+08,740.4,0.765484,0.715909
4,5,The Walking Dead,TV Show,2010,Horror,738600000,738.6,the walking dead,The Walking Dead,2010,...,Won 2 Primetime Emmys. 86 wins & 237 nominatio...,English,United States,https://m.media-amazon.com/images/M/MV5BYWQwMG...,16,20.420267,4.344706e+07,738.6,0.763623,0.784091


In [140]:
# Top 10 shows based on standardised imdb rating
enriched_df[['title','imdb_rating_zscore']].sort_values(by='imdb_rating_zscore', ascending=False).head(10)

,title,imdb_rating_zscore
11491,Brainstorm,1.000000
671,Intersection,0.954545
13,Breaking Bad,0.943182
13862,Unconditional,0.943182
12330,Ayaanle,0.931818
13138,Flavours of Romania,0.931818
116,Avatar: The Last Airbender,0.920455
4382,The Shawshank Redemption,0.920455
6327,Atelier,0.920455
7484,The Chaos Class,0.909091


In [141]:
enriched_df.columns.to_list()

['new_rank',
 'title',
 'type',
 'premiere',
 'genre',
 'total_watchtime',
 'total_watchtime_million',
 'title_clean',
 'imdb_title',
 'year',
 'released_date',
 'rated',
 'imdb_genre',
 'director',
 'writer',
 'actors',
 'plot',
 'imdb_type',
 'total_seasons',
 'imdb_rating',
 'imdb_votes',
 'runtime_minutes',
 'awards',
 'language',
 'country',
 'poster',
 'years_since_release',
 'log_watchtime',
 'recency_weighted_watchtime',
 'evergreen_score',
 'watchtime_zscore',
 'imdb_rating_zscore']

In [183]:
enriched_df.shape

(14647, 32)

In [189]:
aggregated_df['title_clean'] = clean_title(aggregated_df['title'])
aggregated_df.head()

,new_rank,title,type,premiere,genre,total_watchtime,total_watchtime_million,title_clean
0,1,Ginny & Georgia,TV Show,2021,Drama,967200000,967.2,ginny and georgia
1,2,The Night Agent,TV Show,2023,Action,812100000,812.1,the night agent
2,3,You,TV Show,2018,Crime,766300000,766.3,you
3,4,Outer Banks,TV Show,2020,Drama,740400000,740.4,outer banks
4,5,The Walking Dead,TV Show,2010,Horror,738600000,738.6,the walking dead


In [195]:
df_with_year.columns.to_list()

['rank',
 'title',
 'type',
 'premiere',
 'genre',
 'watchtime',
 'watchtime_in_million',
 'title_clean']

In [196]:
trimmed_df_with_year= df_with_year[['rank', 'premiere', 'watchtime','watchtime_in_million','title_clean']]
trimmed_df_with_year = trimmed_df_with_year.drop_duplicates()

In [197]:
trimmed_df_with_year.shape

(18030, 5)

In [198]:
enriched_df['premiere'] = enriched_df['premiere'].astype(str)

In [199]:
final_enriched_df = trimmed_df_with_year.merge(enriched_df, left_on=['title_clean', 'premiere'], right_on=['title_clean', 'premiere'], how='right')
final_enriched_df.head()

,rank,premiere,watchtime,watchtime_in_million,title_clean,new_rank,title,type,genre,total_watchtime,...,awards,language,country,poster,years_since_release,log_watchtime,recency_weighted_watchtime,evergreen_score,watchtime_zscore,imdb_rating_zscore
0,2,2021,665100000,665.1M,ginny and georgia,1,Ginny & Georgia,TV Show,Drama,967200000,...,Nominated for 1 Primetime Emmy. 5 wins & 8 nom...,English,"United States, Canada",https://m.media-amazon.com/images/M/MV5BNGY4Nj...,5,20.689916,1.612000e+08,967.2,1.000000,0.715909
1,9,2021,302100000,302.1M,ginny and georgia,1,Ginny & Georgia,TV Show,Drama,967200000,...,Nominated for 1 Primetime Emmy. 5 wins & 8 nom...,English,"United States, Canada",https://m.media-amazon.com/images/M/MV5BNGY4Nj...,5,20.689916,1.612000e+08,967.2,1.000000,0.715909
2,1,2023,812100000,812.1M,the night agent,2,The Night Agent,TV Show,Action,812100000,...,2 wins & 8 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMDU5Zj...,3,20.515134,2.030250e+08,812.1,0.839624,0.704545
3,6,2018,440600000,440.6M,you,3,You,TV Show,Crime,766300000,...,4 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMTVlYm...,8,20.457084,8.514444e+07,766.3,0.792266,0.727273
4,70,2018,123500000,123.5M,you,3,You,TV Show,Crime,766300000,...,4 wins & 11 nominations total,English,United States,https://m.media-amazon.com/images/M/MV5BMTVlYm...,8,20.457084,8.514444e+07,766.3,0.792266,0.727273


In [200]:
final_enriched_df.shape

(18085, 35)

In [203]:
#final_enriched_df.to_csv('data/final.csv', index = False)
enriched_df.drop(columns = ['title_clean'], inplace = True)

In [204]:
# Write dataframe to Excel file
with pd.ExcelWriter("data/Processed Dacoe Netflix Data.xlsx") as writer:
    netflix_df.to_excel(writer, sheet_name="Original Data", index = False)
    df_with_year.to_excel(writer, sheet_name="Premiere Year Present", index = False)
    df_without_year.to_excel(writer, sheet_name="No Premiere Year", index = False)
    enriched_df.to_excel(writer, sheet_name="Additional IMDb Data", index = False)